# Encoder swap - does TxGNN need the R-GCN?

The KG, every relation, the splits and the metric-learning module are held **completely fixed**. The only
thing that changes is the encoder that produces node embeddings for the DistMult decoder.

| arm | encoder | message passing | relation-specific weights | `proto` | encoder params |
|---|---|---|---|---|---|
| `rgcn` | `HeteroRGCNLayer` x2 (library default) | yes | yes, one `Linear` per relation | True | ~1,010,000 |
| `light_gnn` | `SharedRelationLayer` x2 | yes | **no**, one `Linear` shared by all 50 relations | True | ~20,200 |
| `mlp` | `MLPLayer` x2 | **no** | no | True | ~20,200 |
| `mlp_no_proto` | `MLPLayer` x2 | **no** | no | **False** | ~20,200 |

The three encoders have identical depth (2 layers), width (100) and nonlinearity (`leaky_relu`, applied by
`HeteroRGCN.forward`), and identical aggregation where aggregation exists. So:

- **`rgcn` vs `light_gnn`** isolates relation-specific parameterisation - a 50x cut in encoder parameters.
- **`light_gnn` vs `mlp`** isolates message passing *exactly*. Same parameter count, same shapes; the only
  difference is whether `scatter`-mean over `edge_index` happens.
- **`mlp` vs `mlp_no_proto`** isolates metric learning with the encoder already stripped - the floor.

### Where the KG still enters the `mlp` arms

This matters for reading the result honestly. With `MLPLayer`, no graph structure reaches the node
embeddings. The KG still enters the model in three other places, all untouched:

1. **The metric-learning signature.** `obtain_disease_profile` reads `G.edge_index` directly to build each
   disease's binary neighbour vector over `disease_disease` + `rev_disease_protein`, and `sim_matrix` turns
   that into the similarity used to pick the `proto_num=3` diseases pooled into `h_i_hat`. This is the
   main surviving channel, and it is the one `mlp_no_proto` removes.
2. **The DistMult decoder** - one learned relation vector per edge type (`w_rels`, 50 x 100).
3. **Training and negative sampling** - the supervised edges and `Full_Graph_NegSampler` are unchanged.

Also note `initialize_node_embedding` creates node features as **frozen** xavier-random vectors
(`requires_grad=False`, `utils.py:1127`). So in the `mlp` arms a node embedding is a learned low-rank
transform of a fixed random vector - closer to random-feature regression than to a learned per-node
embedding table. That is a feature of the comparison, not a bug: it means any skill the `mlp` arm shows
has to come from the three channels above.

### Resumability

Identical to `ablation_stage_a.ipynb`: each finished run writes into
`results/ablation_encoder_swap/<run_key>/` immediately, runs with both output files are skipped, and the
notebook is safe to re-run from cell 1 after a crash, kernel restart or interrupt.

---
## 1 - Environment

In [1]:
import os, sys, json, time, gc, random, shutil, inspect, platform, warnings
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings('ignore')


def _find_repo_root(start=None):
    p = Path(start or os.getcwd()).resolve()
    for cand in [p, *p.parents]:
        if (cand / 'pyg_implementation' / 'txgnn').is_dir():
            return cand
    raise RuntimeError('Could not find pyg_implementation/txgnn above ' + str(p))


REPO_ROOT = _find_repo_root()
PYG_ROOT  = REPO_ROOT / 'pyg_implementation'

# Put the PyG port FIRST on sys.path so `import txgnn` never resolves to dgl_implementation/.
sys.path.insert(0, str(PYG_ROOT))

import torch
import torch.nn as nn
from torch_scatter import scatter

import txgnn
from txgnn import TxData, TxGNN
from txgnn.model import HeteroRGCN, HeteroRGCNLayer
from txgnn.utils import evaluate_fb, create_pyg_graph, get_n_params

assert 'pyg_implementation' in txgnn.__file__, (
    'Wrong txgnn package on sys.path (%s). Restart the kernel.' % txgnn.__file__)

# The Gumbel-gate switch was removed from the library in commit 2d064cc. Nothing here passes it;
# fail loudly if it ever comes back, rather than silently training a different model.
_MI_PARAMS = set(inspect.signature(TxGNN.model_initialize).parameters)
assert 'use_decision_network' not in _MI_PARAMS, (
    'model_initialize accepts use_decision_network again - revisit MODEL_CFG before trusting this grid.')


def set_all_seeds(seed: int):
    '''Seed every RNG the training path touches.'''
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


DEVICE = 'cuda:0' if torch.cuda.is_available() else 'cpu'

print('repo root     :', REPO_ROOT)
print('txgnn package :', txgnn.__file__)
print('torch         :', torch.__version__)
print('device        :', DEVICE, '|',
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else platform.processor())
if DEVICE == 'cpu':
    print('\nWARNING: CPU only. A ~15 GPU-hour grid is not practical here.')

repo root     : /home/dsls/Desktop/projects/BardiaSabbagh/txgnn-refactor
txgnn package : /home/dsls/Desktop/projects/BardiaSabbagh/txgnn-refactor/pyg_implementation/txgnn/__init__.py
torch         : 2.6.0+cu118
device        : cuda:0 | NVIDIA GeForce RTX 4060 Ti


---
## 2 - Configuration

Model and schedule are identical to `ablation_stage_a.ipynb` and `ablation_signature_only.ipynb`, so all
three grids are directly comparable. `MODEL_CFG['attention']` stays `False` - `AttHeteroRGCNLayer` is a
different published ablation ("TxGAT") and is not part of this comparison.

`SEEDS` runs to 10. Seeds 6-10 have no split on disk yet, so `prepare_split` will create
`data/complex_disease_<seed>/` for each of them the first time it is asked - a few minutes per seed, once.

In [ ]:
SPLIT   = 'complex_disease'
SEEDS   = SEEDS   = [31, 41, 42, 63, 70, 77, 86, 89, 95, 98]
PROFILE = 'full'

MODEL_CFG = dict(
    n_hid       = 100,
    n_inp       = 100,
    n_out       = 100,
    proto       = True,                  # metric learning (arm mlp_no_proto sets False)
    proto_num   = 3,
    attention   = False,
    sim_measure = 'all_nodes_profile',
    agg_measure = 'rarity',
)

PRETRAIN_CFG = dict(n_epoch=1,    learning_rate=1e-3, batch_size=1024, train_print_per_n=500)
FINETUNE_CFG = dict(n_epoch=1000, learning_rate=5e-4, train_print_per_n=100, valid_per_n=20)

DD_REL     = ['contraindication', 'indication', 'off-label use']
DD_REV_REL = ['rev_' + r for r in DD_REL]
DD_ETYPES  = [('drug', r, 'disease') for r in DD_REL] + [('disease', r, 'drug') for r in DD_REV_REL]

SIG_ETYPES_DEFAULT = ['disease_disease', 'rev_disease_protein']

DATA_FOLDER  = REPO_ROOT / 'data'
ABL_ROOT     = REPO_ROOT / 'results' / 'ablation_encoder_swap'
STAGE_A_ROOT = REPO_ROOT / 'results' / 'ablation_stage_a'
ABL_ROOT.mkdir(parents=True, exist_ok=True)

REF_ARM = 'rgcn'          # every delta is measured against this arm

# --- switches ------------------------------------------------------------------------------------
RUN_GRID          = True   # section 6.2 trains; set False to inspect the plan / aggregate only
SAVE_MODELS       = False
IMPORT_REF        = True   # reuse Stage A's baseline_seed1..5 as rgcn_seed1..5 - see section 5
PROBE_ENCODER     = True   # after each run, prove whether the encoder reads edge_index at all
STOP_AFTER_N_RUNS = None   # e.g. 2 for a short unattended smoke pass over the harness

print('split   :', SPLIT)
print('seeds   :', SEEDS)
print('results :', ABL_ROOT)
print('pretrain:', PRETRAIN_CFG)
print('finetune:', FINETUNE_CFG)

split   : complex_disease
seeds   : [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
results : /home/dsls/Desktop/projects/BardiaSabbagh/txgnn-refactor/results/ablation_encoder_swap
pretrain: {'n_epoch': 1, 'learning_rate': 0.001, 'batch_size': 1024, 'train_print_per_n': 500}
finetune: {'n_epoch': 1000, 'learning_rate': 0.0005, 'train_print_per_n': 100, 'valid_per_n': 20}


---
## 3 - The two replacement encoders

Both expose the same interface as `HeteroRGCNLayer`: `forward(G, feat_dict) -> {ntype: tensor}`. That is
all `HeteroRGCN.forward` and `HeteroRGCN.forward_minibatch` require, so `w_rels`,
`DistMultPredictor`, the `rarity` gate, pretraining and fine-tuning are untouched.

`SharedRelationLayer` reproduces `HeteroRGCNLayer`'s aggregation exactly - scatter-mean per edge type,
then sum across edge types, zeros for node types with no incoming edges - and differs **only** in using
one `nn.Linear` for every relation instead of an `nn.ModuleDict` of 50. It applies that weight once per
node type rather than once per edge type, which is also a real speedup.

`MLPLayer` is the same module with the aggregation deleted.

Not supported by either: `graphmask_forward` and `add_graphmask_parameters`. The GraphMask explainer
would raise on these arms; it is not used here.

In [3]:
class SharedRelationLayer(nn.Module):
    '''Lightweight GNN layer: HeteroRGCNLayer's aggregation with ONE weight shared by all relations.'''

    def __init__(self, in_size, out_size):
        super().__init__()
        self.weight = nn.Linear(in_size, out_size)
        self.in_size, self.out_size = in_size, out_size

    def forward(self, G, feat_dict):
        Wh = {nt: self.weight(feat_dict[nt]) for nt in G.node_types}
        out = {nt: None for nt in G.node_types}

        for srctype, etype, dsttype in G.edge_types:
            ei = G[(srctype, etype, dsttype)].edge_index
            if ei.shape[1] == 0:
                continue
            msg = Wh[srctype][ei[0]]
            # scatter mean per etype, then sum across etypes - identical to HeteroRGCNLayer
            agg = scatter(msg, ei[1], dim=0, reduce='mean', dim_size=G[dsttype].num_nodes)
            out[dsttype] = agg if out[dsttype] is None else out[dsttype] + agg

        device = next(iter(feat_dict.values())).device
        for nt in G.node_types:
            if out[nt] is None:
                out[nt] = torch.zeros(G[nt].num_nodes, self.out_size, device=device)
        return out


class MLPLayer(nn.Module):
    '''No message passing: h[ntype] = W x[ntype]. Same parameters and shapes as SharedRelationLayer.'''

    def __init__(self, in_size, out_size):
        super().__init__()
        self.weight = nn.Linear(in_size, out_size)
        self.in_size, self.out_size = in_size, out_size

    def forward(self, G, feat_dict):
        return {nt: self.weight(feat_dict[nt]) for nt in G.node_types}


# --- installing an encoder ------------------------------------------------------------------------
# model_initialize() builds `HeteroRGCN(...)` by looking the name up in the txgnn.TxGNN module globals
# (TxGNN.py does `from .model import *`), so rebinding that attribute swaps the encoder without
# touching the library. `txgnn.TxGNN` the attribute is the *class*, so reach the module via sys.modules.
_TXGNN_MODULE   = sys.modules['txgnn.TxGNN']
_ORIG_HETERORGCN = HeteroRGCN
LAYER_CLASSES    = {'rgcn': HeteroRGCNLayer, 'light_gnn': SharedRelationLayer, 'mlp': MLPLayer}


def make_encoder_class(layer_cls, kind):
    class SwappedHeteroRGCN(_ORIG_HETERORGCN):
        '''HeteroRGCN with layer1/layer2 replaced. Everything else is the library's.'''

        def __init__(self, G, **kw):
            # super() builds the stock R-GCN layers and then we discard them. Cheap (<1 s) but it does
            # consume RNG draws, so the parameter init stream differs from the rgcn arm. That is fine -
            # each arm is seeded independently and is deterministic within itself.
            super().__init__(G, **kw)
            self.layer1 = layer_cls(kw['in_size'], kw['hidden_size'])
            self.layer2 = layer_cls(kw['hidden_size'], kw['out_size'])
            self.encoder_kind = kind

    SwappedHeteroRGCN.__name__ = 'HeteroRGCN_' + kind
    return SwappedHeteroRGCN


def install_encoder(kind):
    '''Point model_initialize at the encoder for `kind`. Returns the class it installed.'''
    if kind not in LAYER_CLASSES:
        raise ValueError('unknown encoder %r' % kind)
    cls = _ORIG_HETERORGCN if kind == 'rgcn' else make_encoder_class(LAYER_CLASSES[kind], kind)
    _TXGNN_MODULE.HeteroRGCN = cls
    return cls


install_encoder('rgcn')     # leave the library default in place until a run asks otherwise
print('encoders available:', sorted(LAYER_CLASSES))
print('txgnn.TxGNN.HeteroRGCN ->', _TXGNN_MODULE.HeteroRGCN.__name__)

encoders available: ['light_gnn', 'mlp', 'rgcn']
txgnn.TxGNN.HeteroRGCN -> HeteroRGCN


---
## 4 - The arms

In [4]:
ARMS = [
    {'arm': 'rgcn',         'encoder': 'rgcn',      'overrides': {}},
    {'arm': 'light_gnn',    'encoder': 'light_gnn', 'overrides': {}},
    {'arm': 'mlp',          'encoder': 'mlp',       'overrides': {}},
    {'arm': 'mlp_no_proto', 'encoder': 'mlp',       'overrides': {'proto': False}},
]

ARMS_BY_NAME = {a['arm']: a for a in ARMS}
assert len(ARMS_BY_NAME) == len(ARMS), 'duplicate arm name'
assert REF_ARM in ARMS_BY_NAME, 'REF_ARM %r is not an arm' % REF_ARM

print('%d arms x %d seeds = %d runs\n' % (len(ARMS), len(SEEDS), len(ARMS) * len(SEEDS)))
for a in ARMS:
    cfg = dict(MODEL_CFG, **a['overrides'])
    mp = 'no' if a['encoder'] == 'mlp' else 'yes'
    print('  %-14s encoder=%-10s message passing=%-4s proto=%s'
          % (a['arm'], a['encoder'], mp, cfg['proto']))

4 arms x 10 seeds = 40 runs

  rgcn           encoder=rgcn       message passing=yes  proto=True
  light_gnn      encoder=light_gnn  message passing=yes  proto=True
  mlp            encoder=mlp        message passing=no   proto=True
  mlp_no_proto   encoder=mlp        message passing=no   proto=False


---
## 5 - Mechanics and plan

**`RunData`** rebuilds `G` from the untouched `df_train` for every run, so no run inherits a graph another
run moved to the GPU or re-embedded. Nothing is filtered in this notebook - every arm sees all 50 edge
types and all 6,735,310 training edges.

**`encoder_reads_graph`** is the proof that the `mlp` arms pass no messages. After training and
evaluation, it runs the encoder twice - once normally, once with every `edge_index` replaced by an empty
tensor - and reports the largest absolute change in the node embeddings. `0.0` means the encoder never
looked at an edge. It runs after metrics are computed, so it cannot perturb training, and it restores
`edge_index` in a `finally` block.

**Importing the reference arm.** `rgcn` is configuration-identical to Stage A's `baseline`. With
`IMPORT_REF = True`, seeds 1-5 are copied from `results/ablation_stage_a/baseline_seed<n>/` after checking
that the recorded `split`, `model_config` (the eight keys this notebook sets), `pretrain`, `finetune`,
`dropped_edges_total == 0` and `signature_disease_etypes` all match. Any mismatch falls through to a
normal run, and imported records carry an `imported_from` field. Imported records predate this notebook's
`encoder_params` field, so section 7 takes that count from whichever `rgcn` run has it.

In [5]:
class RunData:
    '''Minimal TxData stand-in. Graph rebuilt per run; nothing filtered in this notebook.'''

    def __init__(self, base, data_folder, split):
        self.df       = base['df']
        self.df_train = base['df_train']
        self.df_valid = base['df_valid']
        self.df_test  = base['df_test']
        self.G = create_pyg_graph(self.df_train, self.df)
        self.data_folder = str(data_folder)
        self.split = split
        self.disease_eval_idx = None
        self.no_kg = False


_SEED_CACHE = {}


def get_seed_data(seed):
    '''Load (and cache) the split frames for one seed. Holds one seed at a time - `df` is huge.'''
    if seed in _SEED_CACHE:
        return _SEED_CACHE[seed]
    _SEED_CACHE.clear()
    gc.collect()
    print('[data] loading split seed %d ...' % seed)
    t0 = time.time()
    set_all_seeds(seed)
    d = TxData(data_folder_path=str(DATA_FOLDER))
    d.prepare_split(split=SPLIT, seed=seed, no_kg=False)
    base = {'df': d.df, 'df_train': d.df_train, 'df_valid': d.df_valid, 'df_test': d.df_test}
    del d
    gc.collect()
    _SEED_CACHE[seed] = base
    print('[data] seed %d ready in %.1f s | train edges %d'
          % (seed, time.time() - t0, len(base['df_train'])))
    return base


def run_dir(arm, seed):
    return ABL_ROOT / ('%s_seed%d' % (arm, seed))


def is_done(arm, seed):
    d = run_dir(arm, seed)
    return (d / 'edge_level_test_metrics.csv').exists() and (d / 'run_meta.json').exists()


def encoder_params(model):
    return (sum(p.numel() for p in model.layer1.parameters())
            + sum(p.numel() for p in model.layer2.parameters()))


def encoder_reads_graph(model, G):
    '''Max |delta| in node embeddings when every edge_index is emptied. 0.0 => no message passing.'''
    model.eval()
    saved = {et: G[et].edge_index for et in G.edge_types}
    empty = torch.zeros((2, 0), dtype=torch.long, device=saved[next(iter(saved))].device)
    try:
        with torch.no_grad():
            h_full = model(G, G, return_h=True)
            for et in G.edge_types:
                G[et].edge_index = empty
            h_none = model(G, G, return_h=True)
        return float(max((h_full[nt] - h_none[nt]).abs().max().item() for nt in h_full))
    finally:
        for et, ei in saved.items():
            G[et].edge_index = ei


# --- import pass, then the plan -------------------------------------------------------------------
MODEL_KEYS_CHECKED = sorted(MODEL_CFG)


def import_matching_run(src, dst, expect, extra=None):
    '''Copy a finished run from another grid iff its metadata matches `expect`. Returns (ok, why).'''
    sm, se = src / 'run_meta.json', src / 'edge_level_test_metrics.csv'
    if not (sm.exists() and se.exists()):
        return False, 'no run at %s' % src.name
    meta = json.loads(sm.read_text())
    for k, want in expect.items():
        got = meta.get(k, '<missing>')
        if isinstance(want, dict):
            if not isinstance(got, dict):
                return False, '%s is not a dict' % k
            bad = {kk: (got.get(kk, '<missing>'), vv) for kk, vv in want.items()
                   if got.get(kk, '<missing>') != vv}
            if bad:
                return False, '%s mismatch %s' % (k, bad)
        elif got != want:
            return False, '%s mismatch: %r != %r' % (k, got, want)

    dst.mkdir(parents=True, exist_ok=True)
    shutil.copyfile(se, dst / 'edge_level_test_metrics.csv')
    meta.update(extra or {})
    meta['imported_from'] = str(src.relative_to(REPO_ROOT))
    meta['run_key'] = dst.name
    meta['arm'] = dst.name.rsplit('_seed', 1)[0]
    with open(dst / 'run_meta.json', 'w') as f:
        json.dump(meta, f, indent=2, default=str)
    return True, 'imported from %s' % src.relative_to(REPO_ROOT)


def try_import_ref(seed):
    if not IMPORT_REF:
        return False, 'disabled'
    return import_matching_run(
        STAGE_A_ROOT / ('baseline_seed%d' % seed),
        run_dir(REF_ARM, seed),
        expect={'split': SPLIT, 'seed': seed,
                'dropped_edges_total': 0,
                'signature_disease_etypes': SIG_ETYPES_DEFAULT,
                'model_config': {k: MODEL_CFG[k] for k in MODEL_KEYS_CHECKED},
                'pretrain': PRETRAIN_CFG, 'finetune': FINETUNE_CFG},
        extra={'encoder': 'rgcn', 'proto': True, 'message_passing': True,
               'experiment': 'encoder_swap'},
    )


if IMPORT_REF:
    for s in SEEDS:
        if is_done(REF_ARM, s):
            continue
        ok, why = try_import_ref(s)
        print('[import] %s_seed%d: %s' % (REF_ARM, s, why if ok else 'will train (%s)' % why))

BASE_PRETRAIN_S, BASE_FINETUNE_S = 1155.7, 316.5
_ref = STAGE_A_ROOT / 'baseline_seed1' / 'run_meta.json'
if _ref.exists():
    _r = json.loads(_ref.read_text())
    BASE_PRETRAIN_S = _r.get('pretrain_seconds', BASE_PRETRAIN_S)
    BASE_FINETUNE_S = _r.get('finetune_seconds', BASE_FINETUNE_S)
EST_MIN = (BASE_PRETRAIN_S + BASE_FINETUNE_S) / 60

rows = []
for a in ARMS:
    done = [s for s in SEEDS if is_done(a['arm'], s)]
    rows.append({'arm': a['arm'], 'encoder': a['encoder'],
                 'message passing': a['encoder'] != 'mlp',
                 'proto': dict(MODEL_CFG, **a['overrides'])['proto'],
                 'done': len(done), 'todo': len(SEEDS) - len(done)})
display(pd.DataFrame(rows).style.hide(axis='index'))

RUN_KEYS = [(a['arm'], s) for s in SEEDS for a in ARMS]      # seed-major, matches the run order
pending  = [k for k in RUN_KEYS if not is_done(*k)]
print('\nrun_keys (%d total, %d done, %d pending):' % (len(RUN_KEYS), len(RUN_KEYS) - len(pending), len(pending)))
for arm, seed in RUN_KEYS:
    print('  [%s] %s_seed%d' % ('x' if is_done(arm, seed) else ' ', arm, seed))
print('\nestimated remaining: %.1f GPU-hours at <= %.1f min/run (the light arms are faster)'
      % (len(pending) * EST_MIN / 60, EST_MIN))
print('order: seed-major - after seed %d finishes, every arm has n=1 and can already be read.' % SEEDS[0])

[import] rgcn_seed1: imported from results/ablation_stage_a/baseline_seed1
[import] rgcn_seed2: imported from results/ablation_stage_a/baseline_seed2
[import] rgcn_seed3: imported from results/ablation_stage_a/baseline_seed3
[import] rgcn_seed4: imported from results/ablation_stage_a/baseline_seed4
[import] rgcn_seed5: imported from results/ablation_stage_a/baseline_seed5
[import] rgcn_seed6: will train (no run at baseline_seed6)
[import] rgcn_seed7: will train (no run at baseline_seed7)
[import] rgcn_seed8: will train (no run at baseline_seed8)
[import] rgcn_seed9: will train (no run at baseline_seed9)
[import] rgcn_seed10: will train (no run at baseline_seed10)


arm,encoder,message passing,proto,done,todo
rgcn,rgcn,True,True,5,5
light_gnn,light_gnn,True,True,0,10
mlp,mlp,False,True,0,10
mlp_no_proto,mlp,False,False,0,10



run_keys (40 total, 5 done, 35 pending):
  [x] rgcn_seed1
  [ ] light_gnn_seed1
  [ ] mlp_seed1
  [ ] mlp_no_proto_seed1
  [x] rgcn_seed2
  [ ] light_gnn_seed2
  [ ] mlp_seed2
  [ ] mlp_no_proto_seed2
  [x] rgcn_seed3
  [ ] light_gnn_seed3
  [ ] mlp_seed3
  [ ] mlp_no_proto_seed3
  [x] rgcn_seed4
  [ ] light_gnn_seed4
  [ ] mlp_seed4
  [ ] mlp_no_proto_seed4
  [x] rgcn_seed5
  [ ] light_gnn_seed5
  [ ] mlp_seed5
  [ ] mlp_no_proto_seed5
  [ ] rgcn_seed6
  [ ] light_gnn_seed6
  [ ] mlp_seed6
  [ ] mlp_no_proto_seed6
  [ ] rgcn_seed7
  [ ] light_gnn_seed7
  [ ] mlp_seed7
  [ ] mlp_no_proto_seed7
  [ ] rgcn_seed8
  [ ] light_gnn_seed8
  [ ] mlp_seed8
  [ ] mlp_no_proto_seed8
  [ ] rgcn_seed9
  [ ] light_gnn_seed9
  [ ] mlp_seed9
  [ ] mlp_no_proto_seed9
  [ ] rgcn_seed10
  [ ] light_gnn_seed10
  [ ] mlp_seed10
  [ ] mlp_no_proto_seed10

estimated remaining: 14.0 GPU-hours at <= 24.0 min/run (the light arms are faster)
order: seed-major - after seed 1 finishes, every arm has n=1 and can a

---
## 6 - The grid

Each finished run writes `edge_level_test_metrics.csv` and `run_meta.json` into
`results/ablation_encoder_swap/<run_key>/`. The metadata records the encoder, the layer class actually
installed, encoder and total parameter counts, `proto`, the `encoder_reads_graph` probe, and
`sorted(G.edge_types)` - which should be the full 50 for every arm.

`MACRO (all relations)` is comparable across arms here: the graph is identical in every arm, so the mean
is taken over the same 50 edge types every time.

In [6]:
def run_one(arm_name, seed, verbose=True):
    '''Train one (arm, seed) and write its results. Returns the edge-level DataFrame.'''
    arm  = ARMS_BY_NAME[arm_name]
    rdir = run_dir(arm_name, seed)
    el_csv, meta_json = rdir / 'edge_level_test_metrics.csv', rdir / 'run_meta.json'

    if el_csv.exists() and meta_json.exists():
        if verbose:
            print('[skip] %s_seed%d -> cached' % (arm_name, seed))
        return pd.read_csv(el_csv)

    if arm_name == REF_ARM:
        ok, why = try_import_ref(seed)
        if ok:
            print('[skip] %s_seed%d -> %s' % (arm_name, seed, why))
            return pd.read_csv(el_csv)

    cfg = dict(MODEL_CFG, **arm['overrides'])

    print('\n' + '#' * 78)
    print('# %s   seed %d   encoder=%s   message passing=%s   proto=%s'
          % (arm_name, seed, arm['encoder'], arm['encoder'] != 'mlp', cfg['proto']))
    print('#' * 78)

    base = get_seed_data(seed)

    set_all_seeds(seed)
    d = RunData(base, DATA_FOLDER, SPLIT)
    etypes_after = sorted('%s|%s|%s' % et for et in d.G.edge_types)

    # the KG must be intact in every arm of this notebook
    rel_names = {et[1] for et in d.G.edge_types}
    missing = [r for r in SIG_ETYPES_DEFAULT if r not in rel_names]
    assert not missing, 'KG is not intact, missing from G: %s' % missing
    print('[graph] train edges %d | edge types %d (KG intact in every arm)'
          % (len(d.df_train), len(etypes_after)))

    installed = install_encoder(arm['encoder'])

    set_all_seeds(seed)
    m = TxGNN(data=d, weight_bias_track=False, exp_name='%s_seed%d' % (arm_name, seed), device=DEVICE)
    m.model_initialize(**cfg)

    assert type(m.model) is installed, 'encoder swap did not take: %s' % type(m.model).__name__
    layer_cls = type(m.model.layer1).__name__
    assert layer_cls == LAYER_CLASSES[arm['encoder']].__name__, \
        'layer1 is %s, expected %s' % (layer_cls, LAYER_CLASSES[arm['encoder']].__name__)

    enc_p, tot_p = encoder_params(m.model), get_n_params(m.model)
    print('[model] %s | layer=%s | encoder params %d | total params %d'
          % (type(m.model).__name__, layer_cls, enc_p, tot_p))

    t0 = time.time()
    set_all_seeds(seed)
    m.pretrain(**PRETRAIN_CFG)
    pre_s = time.time() - t0

    t0 = time.time()
    set_all_seeds(seed)
    m.finetune(**FINETUNE_CFG)
    fin_s = time.time() - t0

    # ---- edge-level test metrics ----
    (auroc_rel, auprc_rel, micro_auroc, micro_auprc, macro_auroc, macro_auprc), test_loss = \
        evaluate_fb(m.best_model, m.g_test_pos, m.g_test_neg, m.G, m.dd_etypes, DEVICE, mode='test')

    el = pd.DataFrame(
        [('%s --%s--> %s' % et, auroc_rel.get(et, np.nan), auprc_rel.get(et, np.nan))
         for et in DD_ETYPES],
        columns=['relation', 'AUROC', 'AUPRC'])
    el.loc[len(el)] = ['MICRO (drug-disease)', micro_auroc, micro_auprc]
    el.loc[len(el)] = ['MACRO (drug-disease)',
                       float(np.nanmean([auroc_rel.get(et, np.nan) for et in DD_ETYPES])),
                       float(np.nanmean([auprc_rel.get(et, np.nan) for et in DD_ETYPES]))]
    el.loc[len(el)] = ['MACRO (all relations)', macro_auroc, macro_auprc]

    rdir.mkdir(parents=True, exist_ok=True)
    el.to_csv(el_csv, index=False)

    # ---- architectural proof, after metrics so it cannot perturb training ----
    graph_sensitivity = None
    if PROBE_ENCODER:
        graph_sensitivity = encoder_reads_graph(m.best_model, m.G)
        print('[probe] max |delta h| with all edges removed: %.6g  (0 => no message passing)'
              % graph_sensitivity)
        # tolerance, not exact equality: the mlp path never reads edge_index so the two forwards are
        # bitwise identical in practice, but a GEMM is not contractually bit-reproducible run to run
        if arm['encoder'] == 'mlp':
            assert graph_sensitivity < 1e-6, \
                'mlp arm still reads the graph (delta %g)' % graph_sensitivity
        else:
            assert graph_sensitivity > 1e-6, \
                '%s arm does not read the graph (delta %g)' % (arm['encoder'], graph_sensitivity)

    meta = {
        'run_key': '%s_seed%d' % (arm_name, seed),
        'arm': arm_name, 'seed': seed, 'split': SPLIT, 'profile': PROFILE,
        'experiment': 'encoder_swap',
        'encoder': arm['encoder'],
        'encoder_class': type(m.model).__name__,
        'layer_class': layer_cls,
        'message_passing': arm['encoder'] != 'mlp',
        'graph_sensitivity': graph_sensitivity,
        'encoder_params': enc_p,
        'n_params': tot_p,
        'proto': cfg['proto'],
        'signature_disease_etypes': SIG_ETYPES_DEFAULT if cfg['proto'] else [],
        'dropped_edges_total': 0,
        'train_edges': len(d.df_train),
        'n_edge_types_after': len(etypes_after),
        'edge_types_after': etypes_after,
        'model_config': m.config,
        'pretrain': PRETRAIN_CFG, 'finetune': FINETUNE_CFG,
        'pretrain_seconds': pre_s, 'finetune_seconds': fin_s,
        'test_bce_loss': float(test_loss),
        'torch': torch.__version__,
        'device': torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu',
        'finished_at': time.strftime('%Y-%m-%dT%H:%M:%S'),
    }
    # written last - is_done() needs both files, so a crash mid-write leaves the run pending
    with open(meta_json, 'w') as f:
        json.dump(meta, f, indent=2, default=str)

    if SAVE_MODELS:
        mdir = REPO_ROOT / 'saved_models' / 'ablation_encoder_swap' / meta['run_key']
        mdir.mkdir(parents=True, exist_ok=True)
        m.save_model(str(mdir))

    ind = auprc_rel.get(('drug', 'indication', 'disease'), np.nan)
    con = auprc_rel.get(('drug', 'contraindication', 'disease'), np.nan)
    print('[done] %s_seed%d | %.1f min | indication AUPRC %.4f | contraindication AUPRC %.4f -> %s'
          % (arm_name, seed, (pre_s + fin_s) / 60, ind, con, rdir.relative_to(REPO_ROOT)))

    # ---- free the GPU and restore the library encoder before the next run ----
    del m, d, auroc_rel, auprc_rel
    install_encoder('rgcn')
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return el

In [7]:
if RUN_GRID:
    todo = [(a['arm'], s) for s in SEEDS for a in ARMS if not is_done(a['arm'], s)]
    if STOP_AFTER_N_RUNS:
        todo = todo[:STOP_AFTER_N_RUNS]

    print('%d run(s) pending of %d total.' % (len(todo), len(ARMS) * len(SEEDS)))
    grid_t0 = time.time()
    for i, (arm_name, seed) in enumerate(todo, 1):
        print('\n>>> %d/%d  %s_seed%d  (elapsed %.1f h)'
              % (i, len(todo), arm_name, seed, (time.time() - grid_t0) / 3600))
        run_one(arm_name, seed)
    print('\nGrid finished %d run(s) in %.1f h.' % (len(todo), (time.time() - grid_t0) / 3600))
else:
    print('RUN_GRID is False - no training. Set it in section 2.')

35 run(s) pending of 40 total.

>>> 1/35  light_gnn_seed1  (elapsed 0.0 h)

##############################################################################
# light_gnn   seed 1   encoder=light_gnn   message passing=True   proto=True
##############################################################################
[data] loading split seed 1 ...
Found local copy...
Found local copy...
Found local copy...
Found saved processed KG... Loading...
Splits detected... Loading splits....
Creating PyG graph....
Done!
[data] seed 1 ready in 28.1 s | train edges 6735310
[graph] train edges 6735310 | edge types 50 (KG intact in every arm)
[model] HeteroRGCN_light_gnn | layer=SharedRelationLayer | encoder params 20200 | total params 25200
Creating minibatch pretraining dataloader...
Start pre-training with #param: 25200
Epoch: 0 Step: 0 LR: 0.00100 Loss 0.7871, Pretrain Micro AUROC 0.5307 Pretrain Micro AUPRC 0.5432 Pretrain Macro AUROC 0.5118 Pretrain Macro AUPRC 0.6200


KeyboardInterrupt: 

---
## 7 - Aggregate

Deltas are paired per seed against `rgcn`, the same statistic Stage A used.

In [ ]:
def _load_all(root):
    recs = []
    for meta_p in sorted(Path(root).glob('*/run_meta.json')):
        el_p = meta_p.parent / 'edge_level_test_metrics.csv'
        if not el_p.exists():
            continue
        meta = json.loads(meta_p.read_text())
        el = pd.read_csv(el_p).set_index('relation')
        for rel, row in el.iterrows():
            recs.append({'arm': meta['arm'], 'seed': meta['seed'], 'relation': rel,
                         'AUROC': row['AUROC'], 'AUPRC': row['AUPRC'],
                         'encoder': meta.get('encoder'), 'proto': meta.get('proto'),
                         'message_passing': meta.get('message_passing'),
                         'encoder_params': meta.get('encoder_params'),
                         'n_params': meta.get('n_params'),
                         'imported': 'imported_from' in meta})
    return pd.DataFrame(recs)


def _ci95(x):
    x = np.asarray(x, dtype=float)
    return 1.96 * x.std(ddof=1) / np.sqrt(len(x)) if len(x) > 1 else np.nan


ROW = {'indication': 'drug --indication--> disease',
       'contraindication': 'drug --contraindication--> disease',
       'micro': 'MICRO (drug-disease)'}

long = _load_all(ABL_ROOT)
summary, wide, ref = None, None, None

if long.empty:
    print('No runs on disk yet - run the grid cell in section 6 first.')
else:
    wide = (long[long.relation.isin(ROW.values())]
            .pivot_table(index=['arm', 'seed'], columns='relation', values='AUPRC')
            .rename(columns={v: k for k, v in ROW.items()}))

    # imported records predate encoder_params, so take the first non-null per arm
    info = (long.groupby('arm')
                .agg(encoder=('encoder', 'first'), proto=('proto', 'first'),
                     message_passing=('message_passing', 'first'),
                     encoder_params=('encoder_params', lambda s: s.dropna().iloc[0] if s.notna().any() else np.nan),
                     n_params=('n_params', lambda s: s.dropna().iloc[0] if s.notna().any() else np.nan)))

    if REF_ARM in wide.index.get_level_values('arm'):
        ref = wide.xs(REF_ARM, level='arm')
    else:
        print('[warn] no %s runs yet - deltas will be NaN.' % REF_ARM)

    out = []
    for arm, g in wide.groupby(level='arm'):
        g = g.droplevel('arm')
        rec = {'arm': arm, 'n_seeds': len(g),
               'encoder': info.loc[arm, 'encoder'], 'proto': info.loc[arm, 'proto'],
               'message_passing': info.loc[arm, 'message_passing'],
               'encoder_params': info.loc[arm, 'encoder_params']}
        for key in ROW:
            rec['%s_AUPRC' % key] = g[key].mean()
            rec['%s_CI' % key] = _ci95(g[key])
            if arm == REF_ARM:
                rec['%s_delta' % key], rec['%s_delta_CI' % key] = 0.0, np.nan
            elif ref is not None:
                paired = (g[key] - ref[key]).dropna()
                rec['%s_delta' % key] = paired.mean() if len(paired) else np.nan
                rec['%s_delta_CI' % key] = _ci95(paired)
                rec['%s_n_paired' % key] = len(paired)
            else:
                rec['%s_delta' % key], rec['%s_delta_CI' % key] = np.nan, np.nan
        out.append(rec)

    order = [a['arm'] for a in ARMS]
    summary = (pd.DataFrame(out)
                 .assign(_o=lambda d: d.arm.map({a: i for i, a in enumerate(order)}))
                 .sort_values('_o').drop(columns='_o').reset_index(drop=True))
    summary.to_csv(ABL_ROOT / 'encoder_swap_summary.csv', index=False)

    cols = ['arm', 'n_seeds', 'encoder', 'proto', 'message_passing', 'encoder_params',
            'indication_AUPRC', 'indication_CI', 'indication_delta', 'indication_delta_CI',
            'contraindication_AUPRC', 'contraindication_CI', 'contraindication_delta',
            'contraindication_delta_CI', 'micro_AUPRC', 'micro_delta']
    print('%d arms, %d runs on disk -> %s\n'
          % (len(summary), len(wide), (ABL_ROOT / 'encoder_swap_summary.csv').relative_to(REPO_ROOT)))
    display(summary[cols].style.format('{:.4f}', subset=cols[6:], na_rep='-')
                         .format({'encoder_params': '{:,.0f}'}).hide(axis='index'))

    short = summary[summary.n_seeds < len(SEEDS)]
    if len(short):
        print('\n[partial] below %d seeds, deltas provisional: %s'
              % (len(SEEDS), ', '.join('%s (n=%d)' % (r.arm, r.n_seeds) for r in short.itertuples())))

### 7.1 - The headline: parameters spent vs AUPRC kept

One row per arm: how much of the R-GCN's encoder you threw away, and how much AUPRC that cost. The
argument lands if `light_gnn` and `mlp` keep most of the AUPRC at ~2% of the encoder parameters, while
`mlp_no_proto` - the same stripped encoder with metric learning also off - does not.

In [ ]:
if summary is None or ref is None:
    print('Need finished %s runs plus at least one other arm.' % REF_ARM)
else:
    s = summary.set_index('arm')
    ref_p = s.loc[REF_ARM, 'encoder_params']
    rows = []
    for arm in [a['arm'] for a in ARMS if a['arm'] in s.index]:
        r = s.loc[arm]
        rows.append({
            'arm': arm,
            'message passing': bool(r['message_passing']) if pd.notna(r['message_passing']) else None,
            'metric learning': bool(r['proto']) if pd.notna(r['proto']) else None,
            'encoder params': r['encoder_params'],
            '% of rgcn params': 100.0 * r['encoder_params'] / ref_p if pd.notna(ref_p) else np.nan,
            'indication AUPRC': r['indication_AUPRC'],
            'indication delta': r['indication_delta'],
            '% of rgcn AUPRC': 100.0 * r['indication_AUPRC'] / s.loc[REF_ARM, 'indication_AUPRC'],
            'contraind. AUPRC': r['contraindication_AUPRC'],
            'contraind. delta': r['contraindication_delta'],
            'n_seeds': int(r['n_seeds']),
        })
    head = pd.DataFrame(rows)
    head.to_csv(ABL_ROOT / 'params_vs_auprc.csv', index=False)
    display(head.style
                .format({'encoder params': '{:,.0f}', '% of rgcn params': '{:.2f}',
                         '% of rgcn AUPRC': '{:.1f}'})
                .format('{:.4f}', subset=['indication AUPRC', 'indication delta',
                                          'contraind. AUPRC', 'contraind. delta'], na_rep='-')
                .hide(axis='index'))
    print('\n-> %s' % (ABL_ROOT / 'params_vs_auprc.csv').relative_to(REPO_ROOT))

In [ ]:
# Per-seed paired deltas - the sign-consistency check a mean hides.
if summary is not None and ref is not None:
    for key in ['indication', 'contraindication']:
        arms = [a for a in summary.arm if a != REF_ARM]
        tbl = pd.DataFrame({a: (wide.xs(a, level='arm')[key] - ref[key]) for a in arms}).T
        seeds_present = list(tbl.columns)
        tbl.columns = ['seed %d' % s_ for s_ in seeds_present]
        tbl['mean'] = tbl.mean(axis=1)
        tbl['losses'] = (tbl[['seed %d' % s_ for s_ in seeds_present]] < 0).sum(axis=1)
        print('\npaired delta AUPRC vs %s - %s  (negative = the swap hurt)' % (REF_ARM, key))
        display(tbl.style.format('{:.4f}', subset=[c for c in tbl.columns if c != 'losses'], na_rep='-'))

---
## 8 - Reading the result

The 2x2 this notebook completes, together with `proto_off` in `ablation_signature_only.ipynb`:

| | metric learning on | metric learning off |
|---|---|---|
| R-GCN encoder | `rgcn` (here) | `proto_off` (signature-only notebook) |
| MLP encoder | `mlp` (here) | `mlp_no_proto` (here) |

If `mlp` lands near `rgcn` while `mlp_no_proto` and `proto_off` both collapse, the supportable claim is:

> On the `complex_disease` split, TxGNN's edge-level AUPRC survives replacing its 1.01M-parameter
> relation-specific R-GCN encoder with a 20k-parameter MLP that performs no message passing at all. What
> it does not survive is removing the disease-similarity metric learning. The KG reaches the prediction
> through the similarity signature, not through the GNN.

What this still does not show, and what each gap needs:

1. **Why pretraining helps, if it does.** Both arms keep the one-epoch self-supervised pass. An arm with
   `pretrain n_epoch=0` would separate "the encoder is useless" from "the encoder is useful but only
   through pretraining". Cheap to add - 20 min less per run, not more.
2. **That the MLP arm is not just memorising degree.** The `rarity` gate `c_i` is a function of the
   disease's drug-disease degree, which is real signal independent of both the encoder and the signature.
   An arm with `agg_measure='avg'` (flat 0.5 gate) would bound how much of the surviving skill is the
   gate rather than the pooled neighbours.
3. **Generalisation beyond one split protocol.** Everything here is `complex_disease`. Disease-area splits
   need the zero-shot filtering described at the bottom of `ablation_stage_a.ipynb`.
4. **Disease-centric ranking metrics.** Edge-level AUPRC uses one sampled negative per positive. The
   paper's headline protocol ranks every drug per disease and is much harsher; a conclusion this strong
   should eventually be checked there.